# 0504 · 01 · Ingesta de CSV y la columna de datos rescatados

En esta demostración vas a ingerir ficheros CSV en Delta Lake con el patrón **CTAS** (`CREATE TABLE AS SELECT`) y la función `read_files()`, y vas a explorar la columna de datos rescatados, **`_rescued_data`**.

## Objetivos de aprendizaje

Al terminar esta lección serás capaz de:

- Ingerir ficheros CSV como tablas Delta con `CREATE TABLE AS SELECT` (CTAS) y la función `read_files()`.
- Definir y aplicar un esquema explícito con `read_files()` para que la ingesta sea consistente y fiable.
- Gestionar e inspeccionar los datos rescatados: los que no encajan en el esquema definido.

## Antes de empezar

1. Ejecuta **`00_preparar_entorno`** (Run all). Deja en el volumen `/Volumes/lab0504/upload/upload` dos carpetas: `sales-csv/` con los cuatro CSV de ventas y `csv-demo-files/` con dos CSV defectuosos.
2. Conecta este notebook a **Serverless**.

Todas las tablas del laboratorio se crean en el schema `lab0504.bronze`:

In [ ]:
%sql
USE CATALOG lab0504;
USE SCHEMA bronze;

SELECT current_catalog() AS catalogo, current_schema() AS schema;

## B. CTAS con `read_files()` para ingerir ficheros CSV

Los ficheros CSV (*Comma-Separated Values*) son un formato de texto sencillo para guardar datos: cada línea es una fila y los valores van separados por un delimitador. Que se llame "separado por comas" no obliga a usar comas: estos ficheros usan `|`.

En esta demostración usarás ficheros CSV del almacenamiento en la nube. Vas a ver cómo ingerirlos en tablas Delta con CTAS y `read_files()`.

### B1. Inspeccionar los ficheros CSV

1. Lista los ficheros CSV de la carpeta `sales-csv`. Confirma que hay 4.

In [ ]:
%sql
LIST '/Volumes/lab0504/upload/upload/sales-csv';

2. Consulta los ficheros CSV directamente por su ruta y fíjate en lo siguiente:
   - Los ficheros incluyen una fila de cabecera con los nombres de las columnas.
   - Las columnas están delimitadas por la barra vertical (`|`).

     Por ejemplo, la primera fila es:
     `"order_id"|"email"|"transactions_timestamp"|"total_item_quantity"|"purchase_revenue_in_usd"|"unique_items"|"items"`

     Cada valor va entre comillas dobles, y la `|` separa las columnas: hay siete, **order_id, email, transactions_timestamp, total_item_quantity, purchase_revenue_in_usd, unique_items** e **items**.

   La consulta ``csv.`ruta` `` usa las opciones por defecto: separador `,` y sin cabecera. Por eso no separa las columnas y la cabecera aparece como una fila más.

In [ ]:
%sql
SELECT *
FROM csv.`/Volumes/lab0504/upload/upload/sales-csv`
LIMIT 5;

3. Ejecuta la celda de abajo para consultar los CSV con las opciones por defecto de `read_files`.

   Revisa el resultado. Fíjate en que los ficheros **no** se han leído bien: las columnas se llaman `_c0`, `_c1`…, la cabecera aparece como una fila de datos y cada valor contiene trozos de línea con `|` y comillas.

   Para arreglarlo hay que pasarle a `read_files()` más opciones.

In [ ]:
%sql
-- Leer los CSV con las opciones por defecto de read_files
SELECT *
FROM read_files(
        '/Volumes/lab0504/upload/upload/sales-csv',
        format => "csv"
      )
LIMIT 5;

### B2. Opciones de CSV en `read_files()`

1. La celda siguiente lee los CSV con `read_files()` y algunas opciones más. Estas son las que usa (lista completa en [opciones de CSV](https://docs.databricks.com/aws/en/query/formats/csv)):

   - El primer argumento es la ruta de los ficheros CSV.
   - `format => "csv"`: los ficheros están en formato CSV.
   - `sep => "|"`: las columnas van separadas por la barra vertical (`|`).
   - `header => true`: la primera fila contiene los nombres de las columnas.
   - Aquí usas CSV, pero `read_files()` también lee otros formatos, como JSON o Parquet.

   Ejecuta la celda y revisa el resultado. Ahora los CSV se leen bien, y aparece una columna nueva llamada **_rescued_data**.

   **NOTA:** `read_files()` añade automáticamente la columna **_rescued_data** para recoger cualquier dato que no encaje con el esquema inferido o indicado. Aquí todos los ficheros son correctos, así que vale `NULL` en todas las filas.

In [ ]:
%sql
-- Indicar las opciones de CSV en read_files
SELECT *
FROM read_files(
        '/Volumes/lab0504/upload/upload/sales-csv',
        format => "csv",
        sep => "|",
        header => true
      )
LIMIT 5;

2. Ahora que ya lees bien los CSV con `read_files()`, usa un CTAS (`CREATE TABLE AS SELECT`) para:
   - Crear una tabla Delta llamada **sales_bronze**.
   - Añadirle la hora de ingesta y columnas de metadatos de la ingesta, como en el laboratorio 0503:
     - **Hora de ingesta:** [`current_timestamp()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/current_timestamp) devuelve la hora al empezar la consulta; sirve para registrar cuándo se ingirieron los datos.
     - **Columnas de metadatos:** la columna oculta [`_metadata`](https://docs.databricks.com/aws/en/ingestion/file-metadata-column), disponible en todos los formatos de entrada, da acceso a atributos de cada fichero:
       - `_metadata.file_modification_time`: la hora de última modificación del fichero.
       - `_metadata.file_name`: el nombre del fichero.

   Ejecuta la celda y revisa el resultado: la tabla **sales_bronze** se ha creado con los datos de los CSV y las columnas de metadatos nuevas.

In [ ]:
%sql
-- Crear una tabla con read_files
CREATE OR REPLACE TABLE sales_bronze AS
SELECT
  *,
  _metadata.file_modification_time AS file_modification_time,
  _metadata.file_name AS source_file,
  current_timestamp() as ingestion_time
FROM read_files(
        '/Volumes/lab0504/upload/upload/sales-csv',
        format => "csv",
        sep => "|",
        header => true
      );

SELECT *
FROM sales_bronze
LIMIT 10;

   Comprueba que se han cargado los cuatro ficheros: **12.484 filas** en total (3.149 + 2.932 + 3.200 + 3.203).

In [ ]:
%sql
SELECT source_file, count(*) AS filas
FROM sales_bronze
GROUP BY source_file
ORDER BY source_file;

3. Revisa los tipos de las columnas de la tabla **sales_bronze**. Fíjate en que `read_files()` infiere el esquema automáticamente cuando no se le indica uno.

   Deberías ver algo así (algún tipo numérico puede variar, por ejemplo `INT` o `BIGINT`):

   | Columna | Tipo inferido |
   |---|---|
   | order_id | INT |
   | email | STRING |
   | transactions_timestamp | BIGINT |
   | total_item_quantity | INT |
   | purchase_revenue_in_usd | DOUBLE |
   | unique_items | INT |
   | items | STRING |
   | _rescued_data | STRING |
   | file_modification_time | TIMESTAMP |
   | source_file | STRING |
   | ingestion_time | TIMESTAMP |

   Fíjate en `items`: llega como texto aunque contiene JSON, y el importe como `DOUBLE`, no como decimal. La inferencia acierta "lo suficiente", no necesariamente lo que quieres: de ahí el interés de definir un esquema explícito, que verás en la sección C.

   **NOTA:** cuando no se indica el esquema, `read_files()` intenta inferir un esquema común a todos los ficheros encontrados, lo que obliga a leerlos todos salvo que se use `LIMIT`. Incluso con `LIMIT` puede leer más ficheros de los necesarios para obtener un esquema representativo.
   - [Inferencia de esquema](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files)

In [ ]:
%sql
DESCRIBE TABLE EXTENDED sales_bronze;

### B3. (BONUS) Equivalente en Python

Con `spark.read`, la columna de rescate **no** aparece sola: hay que pedirla con la opción `rescuedDataColumn` (es una opción de Databricks).

In [ ]:
df = (spark
      .read
      .option("header", True)
      .option("sep", "|")
      .option("rescuedDataColumn", "_rescued_data")       # <---------- Añade la columna de datos rescatados
      .csv("/Volumes/lab0504/upload/upload/sales-csv")
     )

df.display()

## C. Resolver problemas habituales con CSV

1. Primero, explora el volumen con los ficheros en bruto:

   a. En la barra lateral, selecciona el icono de **Catalog**.

   b. Despliega el catálogo **lab0504**.

   c. Despliega el schema **upload**.

   d. Despliega **Volumes** y después el volumen **upload**. Contiene dos carpetas: `sales-csv`, la que has usado hasta ahora, y **`csv-demo-files`**, la de esta sección.

   e. Despliega la carpeta **csv-demo-files**. Contiene dos ficheros:
      - **malformed_example_1_data.csv**
      - **malformed_example_2_data.csv**

   **NOTA:** para no escribir rutas a mano, haz clic derecho sobre un fichero o carpeta del volumen y elige **Copy volume path**.

In [ ]:
%sql
LIST '/Volumes/lab0504/upload/upload/csv-demo-files';

### C1. Definir un esquema durante la ingesta

Quieres leer el fichero CSV en la tabla Bronze con un esquema definido.

**Ventajas de un esquema explícito:**

- Reduce el riesgo de que el esquema inferido sea inconsistente, sobre todo con datos semiestructurados como JSON o CSV.
- La lectura y carga son más rápidas: Spark aplica directamente los tipos y la estructura, sin tener que inferirlos.
- Mejora el rendimiento con datos grandes, porque reduce mucho el trabajo de cómputo.

1. La consulta de abajo lee el fichero **malformed_example_1_data.csv** como texto, para ver su contenido tal cual.

   Ejecútala y fíjate en lo siguiente:
   - El CSV está delimitado por **|**.
   - El CSV tiene cabecera.

   **NOTA:** la columna **transactions_timestamp** contiene el texto *aaa* en la primera fila, lo que dará problemas cuando intentes leer **transactions_timestamp** como BIGINT.

In [ ]:
%sql
-- Ver el CSV como texto
SELECT *
FROM text.`/Volumes/lab0504/upload/upload/csv-demo-files/malformed_example_1_data.csv`;

2. Usa `read_files` para ver cómo se lee este CSV. Ejecuta la celda y revisa el resultado.

   **IMPORTANTE:** el valor mal formado *aaa* de **transactions_timestamp** hace que la columna se infiera como **STRING**. Pero quieres que **transactions_timestamp** llegue a la tabla Bronze como BIGINT.

In [ ]:
%sql
-- read_files sin esquema
SELECT *
FROM read_files(
        '/Volumes/lab0504/upload/upload/csv-demo-files/malformed_example_1_data.csv',
        format => "csv",
        sep => "|",
        header => true
      );

3. Puedes definir un esquema en `read_files()` para leer los datos con una estructura concreta.

   a. Usa la opción `schema` para definirlo. En este caso:
      - **order_id** como INT
      - **email** como STRING
      - **transactions_timestamp** como BIGINT

   b. Usa la opción `rescuedDataColumn` para recoger en una columna aparte todos los datos que no se pueden leer por no coincidir el tipo, y así revisarlos.

   Ejecuta la celda y revisa el resultado. La fila 1 (*aaa*) no se ha podido leer con el esquema definido: su `transactions_timestamp` queda `NULL` y el valor original se guarda en **_rescued_data**. Conservar las filas que no encajan con el esquema te permite inspeccionarlas y tratarlas después, en lugar de perder datos.

   > **¿Por qué hay que pedir la columna aquí?** Cuando `read_files` **infiere** el esquema, añade `_rescued_data` sola. Cuando le **das** el esquema, no: hay que pedirla con `rescuedDataColumn`. Sin ella, el *aaa* desaparecería sin avisar y solo quedaría un `NULL`.

   **NOTA:** definir el esquema en `read_files` mejora el rendimiento porque se salta el paso costoso de inferirlo, y garantiza una lectura consistente y fiable. Es especialmente útil con datos grandes o semiestructurados.

In [ ]:
%sql
-- Definir un esquema en read_files
SELECT *
FROM read_files(
        '/Volumes/lab0504/upload/upload/csv-demo-files/malformed_example_1_data.csv',
        format => "csv",
        sep => "|",
        header => true,
        schema => '''
            order_id INT,
            email STRING,
            transactions_timestamp BIGINT''',
        rescuedDataColumn => '_rescued_data'    -- Crea la columna _rescued_data
      );

#### Resumen: la columna de datos rescatados

La columna de datos rescatados garantiza que las filas que no encajan con el esquema se rescaten en lugar de descartarse. Contiene cualquier dato que no se ha podido leer por estos motivos:

| Motivo | Ejemplo |
|---|---|
| La columna no está en el esquema | El fichero trae `coupon` y el esquema no la declara: el valor va a `_rescued_data` en lugar de perderse |
| El tipo no coincide | Lo que acabas de ver: `aaa` en una columna `BIGINT` |
| Mayúsculas distintas | El fichero trae `Email` y el esquema `email` |

### C2. Cabeceras que faltan durante la ingesta

En este ejemplo, al CSV le falta por error el nombre de una columna en la cabecera.

1. Ejecuta la celda de abajo para ver el fichero **malformed_example_2_data.csv**. Fíjate en que la primera fila contiene la cabecera, pero falta el nombre de la primera columna (`order_id`): la línea empieza directamente por `|`.

In [ ]:
%sql
-- Ver el CSV como texto
SELECT *
FROM text.`/Volumes/lab0504/upload/upload/csv-demo-files/malformed_example_2_data.csv`;

2. Intenta crear una tabla con el fichero **malformed_example_2_data.csv** y la función `read_files()`. Ejecuta la celda y revisa el resultado.

   Fíjate en lo siguiente:
   - La primera columna del CSV no se ha leído como una columna normal: se ha rescatado en **_rescued_data**.
   - La columna **_rescued_data** guarda los datos rescatados como un texto en formato JSON.
   - Si lo inspeccionas, verás que la columna sin nombre aparece con la clave **_c0**, que contiene su valor como texto, junto con una clave **_file_path**.

In [ ]:
%sql
-- Crear una tabla Delta ingiriendo el CSV
CREATE OR REPLACE TABLE demo_4_example_2_bronze AS
SELECT *
FROM read_files(
        '/Volumes/lab0504/upload/upload/csv-demo-files/malformed_example_2_data.csv',
        format => "csv",
        sep => "|",
        header => true
      );

-- Ver la tabla
SELECT *
FROM demo_4_example_2_bronze;

3. La columna **_rescued_data** es un texto en formato JSON. No vas a ver aquí en detalle cómo tratar este tipo de datos: se trabaja en la demo y el laboratorio de JSON, más adelante.

   Aun así, conviene saber que puedes extraer valores de **_rescued_data** y añadirlos a la tabla. Por ejemplo, para obtener el valor del campo **_c0** puedes usar la sintaxis `_rescued_data:_c0`, como en la celda siguiente.

   `:` devuelve siempre texto, así que se convierte con `cast(… AS BIGINT)`.

   **NOTA:** la celda siguiente devuelve **order_id** como la columna rescatada, delante del resto de columnas. El valor sigue también dentro de `_rescued_data`: la consulta lo copia, no lo saca del JSON.

In [ ]:
%sql
-- Arreglar la columna de datos rescatados
SELECT
  cast(_rescued_data:_c0 AS BIGINT) AS order_id,
  *
FROM read_files(
        '/Volumes/lab0504/upload/upload/csv-demo-files/malformed_example_2_data.csv',
        format => "csv",
        sep => "|",
        header => true
      );

## Comprueba lo que has aprendido

1. **¿Por qué ``csv.`ruta` `` no separa bien las columnas?** Usa las opciones por defecto: separador `,` y sin cabecera. Estos ficheros usan `|` y tienen cabecera.
2. **¿Cuándo aparece `_rescued_data` sola y cuándo hay que pedirla?** `read_files` la añade sola cuando infiere el esquema. Con un esquema explícito, o con `spark.read`, hay que pedirla con `rescuedDataColumn`.
3. **¿Qué pasa con la fila del *aaa* al leer con esquema?** No se pierde: su timestamp queda `NULL` y el valor original se guarda en `_rescued_data`.
4. **¿Por qué definir un esquema explícito si la inferencia funciona?** Es más rápido, no depende de lo que traigan los ficheros y garantiza los tipos que tú quieres.
5. **¿Cómo recuperas una columna que acabó en `_rescued_data`?** Con la sintaxis `:` (`_rescued_data:_c0`) y un `cast` al tipo correcto.

## Volver a empezar

Ejecuta `00_preparar_entorno`: borra las tablas y deja los ficheros como al principio. Las tablas no se borran al terminar este notebook, para que puedas seguir inspeccionándolas.

## Recursos adicionales

- [read_files](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files)
- [Opciones de CSV](https://docs.databricks.com/aws/en/query/formats/csv)
- [Columna de datos rescatados](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema)
- [Columna `_metadata` de ficheros](https://docs.databricks.com/aws/en/ingestion/file-metadata-column)